In [1]:
!pip install pyspark

In [3]:
from google.colab import files
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesAnalysis").getOrCreate()
# Browse and select multiple CSV files
uploaded = files.upload()
# Load selected CSV files into PySpark DataFrames
dataframes = {}
for filename in uploaded.keys():
    df = spark.read.csv(filename, header=True, inferSchema=True)
    dataframes[filename] = df
    print(f"\nFile: {filename}")
    print("Records:", df.count())
    df.show(5)

Saving lazy_execution_stores (1).csv to lazy_execution_stores (1).csv
Saving lazy_execution_sales_25000 (1).csv to lazy_execution_sales_25000 (1) (1).csv
Saving lazy_execution_products (1).csv to lazy_execution_products (1).csv

File: lazy_execution_stores (1).csv
Records: 8
+--------+---------+------+-----------+
|store_id|     city|region| store_type|
+--------+---------+------+-----------+
|     S01|Hyderabad| South|       Mall|
|     S02|Bengaluru| South|High Street|
|     S03|  Chennai| South|       Mall|
|     S04|   Mumbai|  West|High Street|
|     S05|     Pune|  West|       Mall|
+--------+---------+------+-----------+
only showing top 5 rows

File: lazy_execution_sales_25000 (1) (1).csv
Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+----

In [4]:
from pyspark.sql import functions as F
from pyspark import StorageLevel
import time

# Find the sales dataset from the uploaded files
sales_df = None

for filename, df in dataframes.items():
    if {"order_id", "quantity", "unit_price", "order_status"}.issubset(df.columns):
        sales_df = df
        print("Sales file selected:", filename)
        break

if sales_df is None:
    raise ValueError("Sales CSV not found. Please upload the sales dataset.")

print("Total Sales Records:", sales_df.count())
sales_df.show(5)


Sales file selected: lazy_execution_sales_25000 (1) (1).csv
Total Sales Records: 25000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000005|2026-03-14|     S06|      P007|       2|     79999|           0|          Cash|   Co

In [5]:
completed_df = sales_df.filter(
    F.col("order_status") =="completed"
)
print("Transformation created")
print("complete orders:", completed_df.count())
completed_df.show()



Transformation created
complete orders: 0
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+



In [6]:
completed_df.cache()
print("Is Cached:", completed_df.is_cached)
completed_df.count()
completed_df.show()
completed_df.groupBy("product_id").count().show()

Is Cached: True
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+

+----------+-----+
|product_id|count|
+----------+-----+
+----------+-----+



In [7]:
high_quantity_df = sales_df.filter(
    F.col("quantity") > 5
)
high_quantity_df.persist(
    StorageLevel.MEMORY_AND_DISK
)
high_quantity_df.count()
high_quantity_df.show()
print("high_quantity_df.StorageLevel")


+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+
| O000002|2026-05-29|     S07|      P014|       6|     29999|           0|          Card|   Completed|       Online|
| O000003|2026-06-27|     S06|      P008|       6|     22999|           0|          Card|   Completed|     In Store|
| O000004|2026-04-10|     S03|      P008|       7|     22999|          15|          Cash|   Completed|       Online|
| O000006|2026-04-05|     S08|      P005|       8|      2499|           0|           UPI|   Completed|     In Store|
| O000007|2026-02-27|     S03|      P010|       8|      3999|          10|           UPI|   Completed|       Online|
| O000008|2026-03-21|     S07|      P002|       9|      2999|   

In [9]:
print("Before:", completed_df.is_cached)
completed_df.unpersist(blocking=True)
print("After:", completed_df.is_cached)
high_quantity_df.unpersist(blocking=True)

Before: True
After: False


DataFrame[order_id: string, order_date: date, store_id: string, product_id: string, quantity: int, unit_price: int, discount_pct: int, payment_method: string, order_status: string, sales_channel: string]

In [10]:
# Generate 40 batch IDs (0 to 39)
batch_df = spark.range(40).select(
    F.col("id").alias("batch_id")
)

batch_df.show(5)

# Combine each sales record with all 40 batch IDs
large_df = sales_df.crossJoin(batch_df)

print("Original Records:", sales_df.count())
print("Large Dataset Records:", large_df.count())

large_df.show(5)


+--------+
|batch_id|
+--------+
|       0|
|       1|
|       2|
|       3|
|       4|
+--------+
only showing top 5 rows
Original Records: 25000
Large Dataset Records: 1000000
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
|order_id|order_date|store_id|product_id|quantity|unit_price|discount_pct|payment_method|order_status|sales_channel|batch_id|
+--------+----------+--------+----------+--------+----------+------------+--------------+------------+-------------+--------+
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       0|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       1|
| O000001|2026-06-22|     S04|      P002|       3|      2999|          10|   Net Banking|   Completed|     In Store|       2|
| O000001|2026-06-22|     S04|      P002|       3|      2999|     

In [11]:

# Create transformations
result_df = large_df.filter(
    F.col("quantity") > 3
).withColumn(
    "revenue",
    F.col("quantity") * F.col("unit_price")
).groupBy("product_id").agg(
    F.sum("revenue").alias("total_revenue")
)

# First execution
start = time.perf_counter()

result_df.collect()

time1 = time.perf_counter() - start

# Second execution
start = time.perf_counter()

result_df.collect()

time2 = time.perf_counter() - start

print("First Execution:", round(time1, 3), "seconds")
print("Second Execution:", round(time2, 3), "seconds")

First Execution: 2.033 seconds
Second Execution: 0.177 seconds


**Partition**

In [12]:
print("Total Records:", sales_df.count())
print("Number of Partitions:", sales_df.rdd.getNumPartitions())

Total Records: 25000
Number of Partitions: 1


In [13]:
partition_df = sales_df.repartition(8)
print("original Partition:", sales_df.rdd.getNumPartitions())
print("new Partition:", partition_df.rdd.getNumPartitions())
partition_df.count()

original Partition: 1
new Partition: 8


25000

In [18]:
from pyspark.sql import functions as F

# Add partition ID to every record
distribution_df = partition_df.withColumn(
    "partition_id",
    F.spark_partition_id()
)
# Count records in each partition
distribution_df.groupBy("partition_id") \
    .count() \
    .orderBy("partition_id") \
    .show()

+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 3125|
|           1| 3125|
|           2| 3125|
|           3| 3125|
|           4| 3125|
|           5| 3125|
|           6| 3125|
|           7| 3125|
+------------+-----+



In [22]:
store_partitoned_df = sales_df.repartition(
    8, "store_id"
)
print("Partitions:", store_partitoned_df.rdd.getNumPartitions())
store_partitoned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy("partition_id", "store_id"
).count().orderBy("partition_id", "store_id"
).show(30)

Partitions: 8
+------------+--------+-----+
|partition_id|store_id|count|
+------------+--------+-----+
|           0|     S01| 3180|
|           1|     S02| 3121|
|           1|     S08| 3122|
|           3|     S06| 3033|
|           4|     S05| 3117|
|           4|     S07| 3209|
|           5|     S03| 3132|
|           5|     S04| 3086|
+------------+--------+-----+



**Repartition & Coalesc**

In [24]:
df8 = sales_df.repartition(8)
df_repartition = df8.repartition(3)
df_coalesc = df8.coalesce(3)
print("Repartiton:", df_repartition.rdd.getNumPartitions())
print("Caolesce:",df_coalesc.rdd.getNumPartitions())
print("REPARTITION PLAN")
df_repartition.explain()
print("Coalesce PLAN")
df_coalesc.explain()

Repartiton: 3
Caolesce: 3
REPARTITION PLAN
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1247]
         +- FileScan csv [order_id#91,order_date#92,store_id#93,product_id#94,quantity#95,unit_price#96,discount_pct#97,payment_method#98,order_status#99,sales_channel#100] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/lazy_execution_sales_25000 (1) (1).csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<order_id:string,order_date:date,store_id:string,product_id:string,quantity:int,unit_price:...
+- == Initial Plan ==
   Exchange RoundRobinPartitioning(3), REPARTITION_BY_NUM, [plan_id=1247]
   +- FileScan csv [order_id#91,order_date#92,store_id#93,product_id#94,quantity#95,unit_price#96,discount_pct#97,payment_method#98,order_status#99,sales_channel#100] Batched: false, DataFilter

In [25]:
current_partitions = sales_df.rdd.getNumPartitions()
print("Current number of partitions:", current_partitions)

Current number of partitions: 1


In [26]:
partitions_12_df = sales_df.repartition(12)
print("Increased partitions to:", partitions_12_df.rdd.getNumPartitions())

partitions_12_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy("partition_id") \
 .count() \
 .orderBy("partition_id") \
 .show(12)

Increased partitions to: 12
+------------+-----+
|partition_id|count|
+------------+-----+
|           0| 2084|
|           1| 2083|
|           2| 2083|
|           3| 2083|
|           4| 2083|
|           5| 2083|
|           6| 2083|
|           7| 2083|
|           8| 2083|
|           9| 2084|
|          10| 2084|
|          11| 2084|
+------------+-----+



In [27]:
product_partitioned_df = sales_df.repartition(6, "product_id")
print("Partitions using product_id:", product_partitioned_df.rdd.getNumPartitions())
product_partitioned_df.withColumn(
    "partition_id",
    F.spark_partition_id()
).groupBy("partition_id", "product_id") \
 .count() \
 .orderBy("partition_id", "product_id") \
 .show(30)

Partitions using product_id: 6
+------------+----------+-----+
|partition_id|product_id|count|
+------------+----------+-----+
|           0|      P002| 1613|
|           0|      P004| 1574|
|           0|      P012| 1691|
|           0|      P014| 1684|
|           1|      P001| 1743|
|           1|      P003| 1676|
|           2|      P010| 1695|
|           2|      P013| 1737|
|           2|      P015| 1645|
|           3|      P005| 1696|
|           3|      P007| 1602|
|           3|      P008| 1596|
|           3|      P009| 1654|
|           4|      P006| 1678|
|           4|      P011| 1716|
+------------+----------+-----+



In [28]:
coalesced_4_df = partitions_12_df.coalesce(4)
print("Original partitions:", partitions_12_df.rdd.getNumPartitions())
print("Coalesced partitions:", coalesced_4_df.rdd.getNumPartitions())

Original partitions: 12
Coalesced partitions: 4
